In [1]:
from google.colab import auth
from google.cloud import bigquery
from google.colab import drive
from google.colab import files
import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.proportion import proportions_ztest
auth.authenticate_user()
client = bigquery.Client(project="data-analytics-mate")
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Робимо SQL-запит для вивантаження необхідної інформації / making query for download information
query = """
WITH
  session_info AS (
    SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
    FROM `data-analytics-mate.DA.ab_test` ab
    JOIN `data-analytics-mate.DA.session` s
      ON ab.ga_session_id = s.ga_session_id
    JOIN `data-analytics-mate.DA.session_params` sp
      ON ab.ga_session_id = sp.ga_session_id
  ),
  session_with_orders AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT o.ga_session_id) AS session_with_orders
    FROM `data-analytics-mate.DA.order` o
    JOIN session_info
      ON o.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  ),
  events AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name,
      COUNT(ep.ga_session_id) AS event_cnt
    FROM `data-analytics-mate.DA.event_params` ep
    JOIN session_info
      ON ep.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name
  ),
  session AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
    FROM session_info
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  ),
  account AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
    FROM `data-analytics-mate.DA.account_session` acs
    JOIN session_info
      ON acs.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  )
SELECT
  session_with_orders.date,
  session_with_orders.country,
  session_with_orders.device,
  session_with_orders.continent,
  session_with_orders.channel,
  session_with_orders.test,
  session_with_orders.test_group,
  'session_with_orders' AS event_name,
  session_with_orders.session_with_orders AS value
FROM session_with_orders
UNION ALL
SELECT
  events.date,
  events.country,
  events.device,
  events.continent,
  events.channel,
  events.test,
  events.test_group,
  event_name,
  events.event_cnt AS value
FROM events
UNION ALL
SELECT
  session.date,
  session.country,
  session.device,
  session.continent,
  session.channel,
  session.test,
  session.test_group,
  'session' AS event_name,
  session_cnt AS value
FROM session
UNION ALL
SELECT
  account.date,
  account.country,
  account.device,
  account.continent,
  account.channel,
  account.test,
  account.test_group,
  'new account' AS event_name,
  new_account_cnt AS value
FROM account
"""

In [3]:
query_job = client.query(query)
results = query_job.result()

In [4]:
df = results.to_dataframe()
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-02,(not set),mobile,Asia,Direct,2,2,new account,1
1,2020-11-02,(not set),mobile,Europe,Organic Search,2,2,new account,1
2,2020-11-04,(not set),mobile,Europe,Undefined,2,2,new account,1
3,2020-11-21,(not set),desktop,(not set),Organic Search,2,2,new account,1
4,2020-11-23,(not set),desktop,(not set),Direct,2,1,new account,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  dbdate
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  Int64 
 6   test_group  800996 non-null  Int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  Int64 
dtypes: Int64(3), dbdate(1), object(5)
memory usage: 57.3+ MB


In [6]:
df['event_name'].value_counts()

,count
event_name,
session,107210
session_start,106242
page_view,101907
user_engagement,94520
first_visit,81621
scroll,73643
view_promotion,61695
view_item,44869
session_with_orders,25892


In [7]:
# робимо словник з метриками, також сюди можна буде додавати в подальшому додавати інші необхідні метрики / creating a dictionary of metrics; other necessary metrics can also be added here in the future.
metrics_dict = {
    'add_payment_info / session':  ('add_payment_info',  'session'),
    'add_shipping_info / session': ('add_shipping_info', 'session'),
    'begin_checkout / session':    ('begin_checkout',    'session'),
    'new_accounts / session':      ('new account',       'session'),
    'session_with_orders / session': ('session_with_orders', 'session'),
}

CONTROL_GROUP = 1
VARIANT_GROUP = 2

In [8]:
# функція, що перевіряє чи є статистично значущі зміни між двома метриками та розмірцих змін
# this function checks for statistically significant changes between two metrics and the magnitude of those changes

def z_test_proportions(x1, n1, x2, n2):
    p1 = x1 / n1                                            #conversions
    p2 = x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)                          #combined conversion
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2)) #standard deviation
    z = (p1 - p2) / se                                      #z-stat
    p_value = 2 * (1 - norm.cdf(abs(z)))                    #p-value
    change = (p1 - p2) / p2 * 100                           #change metric
    return p1, p2, change, z, p_value

In [9]:
# функція, що розраховує статистичну значущість по метриках зі словника (без сегментів)
#a function that calculates statistical significance for metrics from a dictionary (without segments)

def calculate_significance(df, metrics, segment_cols=None, alpha=0.05):
    segment_cols = segment_cols or []
    index_cols = segment_cols + ['test']


    wide = df.pivot_table(              #make pivot-table
        index=index_cols,
        columns=['test_group', 'event_name'],
        values='value',
        aggfunc='sum',
        fill_value=0
    )

    rows = []


    for idx, r in wide.iterrows():                                    #loop for identify test and segment
        idx = idx if isinstance(idx, tuple) else (idx,)
        keys = dict(zip(index_cols, idx))


        for metric_name, (num_event, den_event) in metrics.items():    #make loop for metrics
            x1 = r.get((VARIANT_GROUP, num_event), 0)
            n1 = r.get((VARIANT_GROUP, den_event), 0)
            x2 = r.get((CONTROL_GROUP, num_event), 0)
            n2 = r.get((CONTROL_GROUP, den_event), 0)

            if n1 == 0 or n2 == 0 or x2 == 0:
                continue

            p1, p2, change, z, p_val = z_test_proportions(x1, n1, x2, n2)

            rows.append({         #write all datas in rows
                'segment_type': ', '.join(segment_cols) if segment_cols else 'total',
                'segment_value': ' | '.join(str(keys[c]) for c in segment_cols) if segment_cols else 'total',
                'test_number': keys['test'],
                'metric': metric_name,
                'numerator_event': num_event,
                'denominator_event': den_event,
                'numerator_count_variant': x1,
                'denominator_count_variant': n1,
                'conversion_rate_variant': p1,
                'numerator_count_control': x2,
                'denominator_count_control': n2,
                'conversion_rate_control': p2,
                'metric_change': change,
                'z_stat': z,
                'p_value': p_val,
                'significant': p_val < alpha,
            })

    return pd.DataFrame(rows)   #make rows to Dataframe and return result of function

In [10]:
result_total = calculate_significance(df, metrics_dict) #перевіряємо наші розрахунки: кількість строк повинна дорівнювати кількості тестів повножену на кількість рядків / checking our results (quantity of rows = quantity metrics * quantity test)

result_total

,segment_type,segment_value,test_number,metric,numerator_event,denominator_event,numerator_count_variant,denominator_count_variant,conversion_rate_variant,numerator_count_control,denominator_count_control,conversion_rate_control,metric_change,z_stat,p_value,significant
0,total,total,1,add_payment_info / session,add_payment_info,session,2229,45193,0.049322,1988,45362,0.043825,12.542021,3.924884,0.000087,True
1,total,total,1,add_shipping_info / session,add_shipping_info,session,3221,45193,0.071272,3034,45362,0.066884,6.560481,2.603571,0.009226,True
2,total,total,1,begin_checkout / session,begin_checkout,session,4021,45193,0.088974,3784,45362,0.083418,6.660587,2.978783,0.002894,True
3,total,total,1,new_accounts / session,new account,session,3681,45193,0.081451,3823,45362,0.084278,-3.354299,-1.542883,0.122859,False
4,total,total,1,session_with_orders / session,session_with_orders,session,4526,45193,0.100148,4514,45362,0.099511,0.640785,0.320049,0.748931,False
5,total,total,2,add_payment_info / session,add_payment_info,session,2409,50244,0.047946,2344,50637,0.046290,3.576911,1.240994,0.214608,False
6,total,total,2,add_shipping_info / session,add_shipping_info,session,3510,50244,0.069859,3480,50637,0.068724,1.650995,0.709557,0.477979,False
7,total,total,2,begin_checkout / session,begin_checkout,session,4313,50244,0.085841,4262,50637,0.084168,1.988164,0.952898,0.340642,False
8,total,total,2,new_accounts / session,new account,session,4184,50244,0.083274,4165,50637,0.082252,1.241934,0.588793,0.556000,False
9,total,total,2,session_with_orders / session,session_with_orders,session,5003,50244,0.099574,5102,50637,0.100756,-1.173410,-0.625388,0.531717,False


In [11]:
#Перевірка коректності групування та валідації даних / Verification of the correctness of data grouping and validation
sessions_raw = df.loc[df['event_name'] == 'session', 'value'].sum() # session in begin data

# session in result data
one_metric = result_total[result_total['metric'] == 'add_payment_info / session']
sessions_result = (one_metric['denominator_count_variant'].sum()
                   + one_metric['denominator_count_control'].sum())

print("Session in begin data:", sessions_raw)
print("Session in result data: ", sessions_result)

Session in begin data: 542142
Session in result data:  542142


In [12]:
# Перевірка коректності розрахунку параметрів z-test / Verification of the correctness of z-test parameter calculations
for _, row in result_total.iterrows():
    z_lib, p_lib = proportions_ztest(
        count=[row['numerator_count_variant'], row['numerator_count_control']],
        nobs=[row['denominator_count_variant'], row['denominator_count_control']]
    )

    assert abs(z_lib - row['z_stat']) < 1e-6, \
        f"z isn't correct: test {row['test_number']}, {row['metric']}"
    assert abs(p_lib - row['p_value']) < 1e-9, \
        f"p-value isn't correct: test {row['test_number']}, {row['metric']}"

print("z-stat та p-value is the same as in statsmodels")

z-stat та p-value is the same as in statsmodels


In [13]:
#знаходимо топ-5 країн за сесіями / identifying the top 5 countries by sessions

top_countries = (df[df['event_name'] == 'session'].groupby('country')['value'].sum().nlargest(5).index.tolist())
print(top_countries)

['United States', 'India', 'Canada', 'United Kingdom', 'France']


In [14]:
# групуємо країни за 5-то та решту / grouping countries into the top 5 and the others

df['country_group'] = np.where(df['country'].isin(top_countries),df['country'],'Other countries')

print(df['country_group'].value_counts())

country_group
Other countries    626134
United States       53507
India               38123
Canada              35165
United Kingdom      26455
France              21612
Name: count, dtype: int64


In [15]:
# задаємо необхідні сегменти / define the necessary segments
SEGMENTS = ['country_group', 'device', 'continent', 'channel']

# створюємо список, де першим значенням є наша таблиця з загальними результатами та додаємо до нього результати в розрізі сегментів розраховані за допомогою функції
# create a list where the first value is our table of overall results, and add to it the results broken down by segment, calculated using the function
all_results = [result_total]
for seg in SEGMENTS:
    all_results.append(calculate_significance(df, metrics_dict, segment_cols=[seg]))
#вертикально з'єднуємо наші таблиці у єдиний датафрейм / vertically join our tables into a single dataframe
final = pd.concat(all_results, ignore_index=True)

# перейменовуємо назву сегменту для подальшої зручності / rename the segment for greater convenience later
final['segment_type'] = final['segment_type'].replace({'country_group': 'country'})


In [16]:
final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   segment_type               420 non-null    object 
 1   segment_value              420 non-null    object 
 2   test_number                420 non-null    int64  
 3   metric                     420 non-null    object 
 4   numerator_event            420 non-null    object 
 5   denominator_event          420 non-null    object 
 6   numerator_count_variant    420 non-null    int64  
 7   denominator_count_variant  420 non-null    int64  
 8   conversion_rate_variant    420 non-null    float64
 9   numerator_count_control    420 non-null    int64  
 10  denominator_count_control  420 non-null    int64  
 11  conversion_rate_control    420 non-null    float64
 12  metric_change              420 non-null    float64
 13  z_stat                     420 non-null    float64

In [17]:
final.tail()

,segment_type,segment_value,test_number,metric,numerator_event,denominator_event,numerator_count_variant,denominator_count_variant,conversion_rate_variant,numerator_count_control,denominator_count_control,conversion_rate_control,metric_change,z_stat,p_value,significant
415,channel,Undefined,4,add_payment_info / session,add_payment_info,session,566,5862,0.096554,496,5716,0.086774,11.270787,1.822812,0.068332,False
416,channel,Undefined,4,add_shipping_info / session,add_shipping_info,session,666,5862,0.113613,640,5716,0.111966,1.470701,0.280026,0.779457,False
417,channel,Undefined,4,begin_checkout / session,begin_checkout,session,1600,5862,0.272944,1578,5716,0.276067,-1.131171,-0.376454,0.706579,False
418,channel,Undefined,4,new_accounts / session,new account,session,510,5862,0.087001,484,5716,0.084675,2.747490,0.446742,0.655061,False
419,channel,Undefined,4,session_with_orders / session,session_with_orders,session,531,5862,0.090583,573,5716,0.100245,-9.637902,-1.769660,0.076784,False


In [18]:
#saving our project to Google Drive for further work on its visualization in Tableau / Зберігаємо наш проєкт на гугл-диск для подальшої роботи над його візуалізацією у Tableau
final.to_csv('/content/drive/My Drive/ab_test_significance_portfolio2.csv', encoding='utf-8-sig', index=False)

#Висновки / Conclusions



У проєкті було розраховано статистичну значущість для чотирьох метрик конверсії (add payment info, add shipping info, begin checkout, new accounts, кожна до session) у чотирьох A/B тестах за допомогою z-тесту для двох пропорцій. Розрахунок реалізовано через цикли та словник метрик, тому нові метрики додаються одним рядком без зміни коду.

Коректність перевірено двічі: сума сесій у результаті збігається з сирими даними (542 142), а z-статистики збігаються з statsmodels.

Окрім загальної, значущість пораховано в розрізах за країнами (топ-5 та Other countries), пристроями, континентами й каналами, а результати об'єднано в одну таблицю.

На дашборді в Tableau (https://public.tableau.com/app/profile/daria.shcherbak7028/viz/portfolio_AB_testing/Dashboard1?publish=yes) поєднано розподіл сесій між групами (перевірка рівності вибірки) і результати значущості, а вибір тесту керується одним параметром для всіх елементів.

Важливо, що значущість (p-value < 0.05) показує лише наявність реальної різниці, а її напрямок визначається знаком зміни метрики: наприклад, у тесті 4 begin_checkout та new_accounts значуще погіршились.

Аналізуючи розраховані показники, можна зазначити, що в першому тесті спостерігається статистично значуще зростання трьох із чотирьох метрик (`add_payment_info`, `add_shipping_info`, `begin_checkout`), тоді як у другому тесті жодна зі змін не є статистично значущою. У третьому та четвертому тестах частина метрик змінилася значуще, а частина ні; зокрема в четвертому тесті значущі зміни `begin_checkout` і `new_accounts` є негативними. Тому для прийняття управлінських рішень потрібно детальніше досліджувати кожен тест у розрізі окремих метрик і сегментів.


Загальний висновок: жоден із варіантів не продемонстрував збільшення кількості покупок, тому впроваджувати жоден із них не варто.
Тест 1: Варіант B показує кращі результати на етапах оформлення замовлення, але не за кількістю покупок.
Користувачі у варіанті B частіше переходять до етапів оплати та доставки (зокрема, фіксується значний приріст за подією `add_payment_info`). Однак кількість замовлень на сесію не змінилася (+0,06). Можливі причини:
- позитивний ефект втрачається на фінальному етапі оплати або підтвердження замовлення;
- додаткові користувачі, які доходять до етапу оформлення замовлення, мають нижчу готовність до покупки й у підсумку нічого не купують.
Рішення: не впроваджувати, оскільки приріст доходу відсутній. Повторити тест (або продовжити його), використовуючи показник «сесії із замовленнями» (`session_with_orders`) як основну метрику.

Тест 2: жодних ознак впливу.
Усі чотири метрики воронки не є статистично значущими (p = 0,22–0,56), а кількість замовлень залишилася на тому ж рівні. Кількість подій додавання в кошик (`add_to_cart`) зросла на 8,9%, проте цей показник не аналізувався в розрізі сесій.
Рішення: залишити варіант A. Відкласти цю ідею; повертатися до неї варто лише в тому разі, якщо буде сформульовано чітку гіпотезу, що обґрунтовує проведення масштабнішого тесту.

Тест 3: у варіанті B користувачі рідше переходять до оформлення замовлення.
Показник початку оформлення замовлення (`begin_checkout`) знизився з 13,61% до 13,15% (−3,35%, p = 0,012). Це єдиний статистично значущий результат у тесті.
Рішення: залишити варіант A. Якщо виникне потреба повернутися до цієї ідеї, слід змінити гіпотезу та використовувати конверсію в покупку як основну метрику.

Тест 4: стабільно негативні результати.
Найбільший за обсягом тест (~105 тис. сесій у кожній групі). Усі чотири метрики воронки у варіанті B виявилися нижчими (від −2,4% до −3,5%); зокрема, зафіксовано значне зниження показників початку оформлення замовлення (`begin_checkout`, p = 0,046) та створення нових облікових записів (`new_accounts`, p = 0,018).
Рішення: залишити варіант A.


In this project, statistical significance was calculated for four conversion metrics—add payment info, add shipping info, begin_checkout, and new accounts (each calculated per session)—across four A/B tests using a two-proportion z-test. The calculation was implemented using loops and a dictionary of metrics, allowing new metrics to be added with a single line of code, without requiring further code modifications.

The accuracy was verified through two checks: the total number of sessions matches the raw data (542,142), and the calculated z-statistics align with those produced by the statsmodels library.

In addition to the overall results, significance was calculated across various segments—countries (top 5 and "Other countries"), devices, continents, and channels—with all findings consolidated into a single table.

A Tableau dashboard (https://public.tableau.com/app/profile/daria.shcherbak7028/viz/portfolio_AB_testing/Dashboard1?publish=yes) integrates the distribution of sessions across groups (to verify sample balance) with the significance results; a single parameter controls the test selection for all dashboard elements.

It is important to note that statistical significance (p-value < 0.05) merely indicates the existence of a genuine difference; the direction of this difference is determined by the sign of the metric's change. For instance, in Test 4, both *begin_checkout* and *new_accounts* showed a statistically significant decline.

Overall: no variant is proven to increase purchases, so none should be rolled out.
 Test 1: B wins on checkout steps, but not on purchases
Users in B reach the payment and shipping steps more often, and the effect is strong: add_payment_info. But orders per session did not change (+0.06). Possible explanations:
- the uplift is lost at the final payment/confirmation step;
- the extra users reaching checkout are lower-intent and do not buy.
Decision: do not roll out as a revenue win. Re-run (or extend) with `session_with_orders` as the primary metric.

Test 2: no evidence of an effect
All four funnel metrics are non-significant (p = 0.22-0.56) and orders are flat. Raw add_to_cart events are +8.9%, but this was not tested per session.
Decision: keep A. Deprioritize this idea, unless a larger test is justified by a clear hypothesis.

Test 3: B starts checkout less often
begin_checkout dropped from 13.61% to 13.15% (−3.35%, p = 0.012). This is the only significant result in the test and it passes the per-test .
Decision: keep A. If the idea is revisited, change the hypothesis and use purchase conversion as the primary metric.

Test 4: consistently negative
The largest test (~105K sessions per group). All four funnel metrics are lower in B (−2.4% to −3.5%), with significant drops in begin_checkout (p = 0.046) and new_accounts (p = 0.018). .
Decision: keep A.